# 06 - Improved XGBoost forecasts and validated allocation
Prediction and decision quality are separate objectives. Compare 14 prespecified profit models on rolling CP04-06 factual RMSE, and 21 feasible allocation rules on rolling DR incremental NCM. Freeze both choices before retrospective CP07 evaluation. CP07 was inspected in the previous audit, so it is not a pristine prospective test.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from src.common import TABLES, FIGS, MODELS, load_processed
from src.modeling import ACTIONS, COMMERCIAL, XGB_FIXED, feature_lists, predict_actions
from src.performance import run_performance_validation
df = load_processed("historical_feature_outcome")
NUM, CAT = feature_lists()
X_COLS = NUM + CAT
outcomes = set(pd.read_csv(TABLES/'outcome_schema.csv').outcome)
assert not set(X_COLS) & (outcomes | {'offer_type','assignment_source','customer_id','campaign_id','send_ts','assignment_id'})
exp = df[df.assignment_source.eq('exploration')]
print(len(exp), 'randomized training rows;', len(X_COLS), 'predecision predictors')

7527 randomized training rows; 43 predecision predictors


## Rebuild the development searches and evaluate frozen choices
The two-part XGBoost family estimates purchase probability times conditional signed buyer margin. It shares data across randomized arms, and tests shrinkage toward arm-specific buyer means. Negative margins remain possible. Selection uses earlier campaigns only. Segmented offer effects, pooled models, action means and simple feasible baselines compete for allocation. A confidence interval excluding zero is not required to select the validation winner; uncertainty is reported separately.

In [2]:
# Keep a local traceback log for long native-library calls; no targets or secrets are logged.
import faulthandler
trace_dir = Path.cwd().parent / '.validation-cache'
trace_dir.mkdir(exist_ok=True)
with (trace_dir/'model_execution_trace.log').open('w') as trace:
    faulthandler.dump_traceback_later(60,repeat=True,file=trace)
    try:
        selection, model_metrics, policy_metrics, final = run_performance_validation(df, NUM, CAT, TABLES, MODELS)
    finally:
        faulthandler.cancel_dump_traceback_later()
forecast = selection['forecast_model']
selected_policy = selection['selected_policy']
print('Forecast model:', forecast, '| Allocation rule:', selected_policy)
display(model_metrics.query("scope == 'pooled'"))
display(policy_metrics[['policy','dr_incremental_ncm_per_customer','ci_low','ci_high','offered_share']])
display(pd.read_csv(TABLES/'policy_candidate_selection.csv'))

CP04 ORIGINAL_TLEARNER 67.7409


CP04 POOLED_STUMPS 67.0898


CP04 POOLED_D2_100 67.0014


CP04 POOLED_D2_200 67.2507


CP04 POOLED_D2_STRONG 67.269


CP04 POOLED_D3_STRONG 67.2253


... (47 outputs left out)

Forecast model: HURDLE_XGB_HALF | Allocation rule: RANDOM_FREE_SHIP


,model,scope,n,rmse,mae,r2,obs_mean,pred_mean
0,ORIGINAL_TLEARNER,pooled,1060,63.369408,27.411143,0.005863,15.31483,16.573692
5,HURDLE_XGB_HALF,pooled,1060,62.738470,26.480589,0.025561,15.31483,15.975588


,policy,dr_incremental_ncm_per_customer,ci_low,ci_high,offered_share
0,ORIGINAL_TLEARNER,3.971406,-2.233164,9.682505,0.6
1,ACTION_MEAN,5.571300,-2.097807,13.623382,0.6
2,HURDLE_XGB_HALF,8.888760,0.252821,18.029904,0.6
3,ALL_NO_OFFER,0.000000,0.000000,0.000000,0.0
4,PROPENSITY_PCT10,3.441699,-2.633536,9.620249,0.6
5,HISTORICAL_MIX,4.659854,-0.483924,9.758685,0.6
6,RANDOM_FREE_SHIP,12.741429,3.589919,22.663654,0.6


,policy,n,development_dr_incremental_ncm
0,RANDOM_FREE_SHIP,3237,3.817569
1,SEGMENT_MEMBERSHIP_HISTORY_RIDGE_100,3237,3.737438
2,SEGMENT_MEMBERSHIP_HISTORY_RIDGE_20,3237,3.737438
3,HISTORICAL_MIX,3237,3.339480
4,ACTION_MEAN,3237,2.768525
5,SEGMENT_MEMBERSHIP_HISTORY_RIDGE_300,3237,2.733803
6,SEGMENT_MEMBERSHIP_CART_RIDGE_100,3237,2.494478
7,SEGMENT_MEMBERSHIP_RIDGE_100,3237,2.051481
8,SEGMENT_MEMBERSHIP_RIDGE_20,3237,2.051481
9,SEGMENT_MEMBERSHIP_RIDGE_300,3237,2.003073


## Save forecast metadata and diagnostics
Legacy `tlearner_ncm_*.joblib` filenames now contain the selected predictive model. The allocation rule is separately recorded in `policy_selection.json`. A random free-shipping rule can be selected even when a more accurate XGBoost forecast is available; accuracy alone does not validate personalized treatment effects.

In [3]:
selected_metrics = model_metrics[model_metrics.model.eq(forecast)]
meta = {'target':'ncm_14d','training_population':'assignment_source == exploration','actions':ACTIONS,
        'numeric_features':NUM,'categorical_features':CAT,'selected_method':forecast,
        'selected_policy':selected_policy,'allocation_model':selection['allocation_model'],'forecast_config':selection['prediction_selection']['config'],
        'xgb_fixed':XGB_FIXED,'xgb_selected':{'n_estimators':100,'max_depth':1,'learning_rate':.03},
        'xgb_selected_role':'reference T-learner / existing cost-model parameters, not the selected hurdle configuration',
        'selection':selection['rule'],'final_train_rows':{a:int(exp.offer_type.eq(a).sum()) for a in ACTIONS},
        'cp07_pooled':selected_metrics.query("scope == 'pooled'").iloc[0][['rmse','mae','r2']].to_dict()}
(MODELS/'tlearner_metadata.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
final_mu = predict_actions(final, exp[X_COLS])
te = pd.read_csv(TABLES/'xgboost_test_predictions.csv')
step04 = pd.read_csv(TABLES/'incremental_ncm_exploration.csv').query("segment == 'exploration' and outcome == 'ncm_14d'").set_index('offer_type')
effects = pd.DataFrame({'action':ACTIONS,
    'model_mean_incremental_ncm_CP07':[te[f'incremental_ncm_{a}'].mean() for a in ACTIONS],
    'final_model_mean_incremental_ncm_all_exploration':(final_mu-final_mu[:,[0]]).mean(axis=0),
    'step04_randomized_effect_CP01_07':[0.]+[step04.loc[a,'incremental_ncm'] for a in COMMERCIAL],
    'interpretation':'model averages are diagnostic, not independent evidence of treatment-effect accuracy'})
effects.to_csv(TABLES/'tlearner_action_effect_summary.csv',index=False)
importance=[]
for component in ['purchase_model','margin_model']:
    model=getattr(final['NO_OFFER'],component,None)
    if model is None or not hasattr(model[-1], 'get_booster'): continue
    names=model[0].get_feature_names_out()
    for key,gain in model[-1].get_booster().get_score(importance_type='gain').items():
        feature=names[int(key[1:])] if key.startswith('f') and key[1:].isdigit() else key
        importance.append({'action':'shared_all_actions','component':component,'encoded_feature':feature,'gain':gain})
pd.DataFrame(importance).to_csv(TABLES/'tlearner_feature_importance.csv',index=False)
display(selected_metrics)

,model,scope,n,rmse,mae,r2,obs_mean,pred_mean
5,HURDLE_XGB_HALF,pooled,1060,62.738470,26.480589,0.025561,15.314830,15.975588
6,HURDLE_XGB_HALF,NO_OFFER,257,41.171565,23.701400,-0.049259,10.244903,18.217000
7,HURDLE_XGB_HALF,PCT_10,294,73.937705,27.207079,0.012394,16.185578,15.672923
8,HURDLE_XGB_HALF,PCT_20,243,35.842004,18.961864,0.089049,11.339588,11.098230
9,HURDLE_XGB_HALF,FREE_SHIP,266,82.654232,35.231391,0.028499,22.882331,18.600169


## Factual prediction error and outcome-based policy value

In [4]:
fig,ax=plt.subplots(1,2,figsize=(14,5))
m=model_metrics.query("scope == 'pooled'")
ax[0].barh(m.model,m.rmse,color='#2a78d6'); ax[0].set_xlabel('CP07 factual RMSE ($)')
pv=policy_metrics.set_index('policy'); x=pv.dr_incremental_ncm_per_customer.to_numpy(); y=np.arange(len(pv))
ax[1].errorbar(x,y,xerr=np.maximum(np.vstack([x-pv.ci_low.to_numpy(),pv.ci_high.to_numpy()-x]),0),fmt='o')
ax[1].set_yticks(y,pv.index); ax[1].axvline(0,color='grey',ls='--'); ax[1].set_xlabel('CP07 constrained DR lift ($/customer)')
fig.tight_layout(); fig.savefig(FIGS/'06_tlearner_diagnostics.png',dpi=120,bbox_inches='tight'); plt.show()

C:\Users\yifan\AppData\Local\Temp\ipykernel_24332\1690949766.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIGS/'06_tlearner_diagnostics.png',dpi=120,bbox_inches='tight'); plt.show()
